In [31]:

import os, sys
import pandas as pd
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
sys.path.append(os.path.abspath(".."))
from src.semantic import semantic_retriever_rag, load_semantic
from src.rag_pipeline import build_context, build_prompt, semantic_rag_pipeline

In [32]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [33]:


llm_endpoint = HuggingFaceEndpoint(
    repo_id = "meta-llama/Meta-Llama-3-8B-Instruct",
    task = "text-generation",
    max_new_tokens = 200,
    huggingfacehub_api_token = hf_token,
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [34]:
test = llm.invoke("Recommend a beginner book about investing in one short paragraph.")
print(test.content)

I recommend "A Random Walk Down Wall Street" by Burton G. Malkiel for beginners. This book provides a comprehensive and accessible introduction to investing, covering the basics of finance, stock markets, and portfolio management. Written in an engaging and non-technical style, it explains complex concepts in simple terms, making it an ideal choice for those new to investing. With over 12 editions, it has become a classic and a reliable resource for anyone looking to start their investment journey.


In [35]:
merged_df = pd.read_parquet("../data/processed/merged.parquet")
semantic_model, semantic_index, semantic_meta = load_semantic()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20666.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
test_docs = semantic_retriever_rag(
    query="good books for beginners learning investing",
    model=semantic_model,
    index=semantic_index,
    df=merged_df,
    top_k=3,
)

test_docs.head()

,parent_asin,product_title,review_title,review_text,rating,semantic_score
3496,B01AR5417A,Investment: A History (Columbia Business Schoo...,Long and tiresome,The book is long and tiresome. I didn't learn ...,3.0,0.491082
3021,B008Y1L7EA,Little Kids Big Money: Tools for Teaching Kid ...,Must Read For Parents,This book really makes a lot of sense on how t...,5.0,0.444927
3609,B005T75AF4,The Node Beginner Book,A good resource for starting Node,"A good resource for starting Node, but it fall...",3.0,0.438229


In [37]:

test_context = build_context(test_docs)
print(test_context)

Product ASIN: B01AR5417A
Title: Investment: A History (Columbia Business School Publishing)
Review Title: Long and tiresome
Rating: 3.0
Review Text: The book is long and tiresome. I didn't learn any interesting theory or facts. The main thesis is that nowadays more people invest while in old ages only the elite invested. But this something that I knew (and I believe everybody else knew) before reading this book.

---

Product ASIN: B008Y1L7EA
Title: Little Kids Big Money: Tools for Teaching Kid Friendly Finance
Review Title: Must Read For Parents
Rating: 5.0
Review Text: This book really makes a lot of sense on how to teach a child about money, how to budget money, and how to really think about money. I love how they talk about different incentives for different types of children. Yes, this author realizes that not all children are the same and what works for one, doesn't work for another.

---

Product ASIN: B005T75AF4
Title: The Node Beginner Book
Review Title: A good resource for st

In [38]:
prompt = build_prompt(
    query="good books for beginners learning investing",
    context=test_context
)

print(prompt)

Use the information below to answer the question clearly and concisely.

Context:
Product ASIN: B01AR5417A
Title: Investment: A History (Columbia Business School Publishing)
Review Title: Long and tiresome
Rating: 3.0
Review Text: The book is long and tiresome. I didn't learn any interesting theory or facts. The main thesis is that nowadays more people invest while in old ages only the elite invested. But this something that I knew (and I believe everybody else knew) before reading this book.

---

Product ASIN: B008Y1L7EA
Title: Little Kids Big Money: Tools for Teaching Kid Friendly Finance
Review Title: Must Read For Parents
Rating: 5.0
Review Text: This book really makes a lot of sense on how to teach a child about money, how to budget money, and how to really think about money. I love how they talk about different incentives for different types of children. Yes, this author realizes that not all children are the same and what works for one, doesn't work for another.

---

Product A

In [39]:
rag_response = llm.invoke(prompt)
print(rag_response.content)

Based on the provided information, there isn't a review for a good book for beginners learning investing. However, one review mentions a book that might be related to investing in general: "Investment: A History (Columbia Business School Publishing)". However, the reviewer found it to be long, tiresome, and didn't learn anything new from it.

A better option for a beginner might be to look for books that discuss personal finance or investing in general. Alternatively, you could consider books that cater to a more specific audience like children learning about money, such as "Little Kids Big Money: Tools for Teaching Kid Friendly Finance". 

If you are looking for good investing books specifically, I can recommend some popular ones. 

 Popular Investing books include:
    - 'A Random Walk Down Wall Street'
    - 'The Intelligent Investor' 
    - 'The Little Book of Common Sense Investing'
    - 'The Essays of Warren Buffett: Lessons for Corporate America'
    - 'Security Analysis'
Howev

In [40]:


result = semantic_rag_pipeline(
    query="good books for beginners learning investing",
    llm=llm,
    model=semantic_model,
    index=semantic_index,
    df=merged_df,
    top_k=3
)

print(result["answer"])

Based on the provided reviews, I would recommend the following books for beginners learning investing:

Unfortunately, the reviews do not directly address books specifically for beginners learning investing. However, based on their titles, I would suggest:

1. "Investment: A History (Columbia Business School Publishing)" - Since the reviewer states the main thesis as something they already knew, it might not be a particularly insightful book for beginners. 
2. "Little Kids Big Money: Tools for Teaching Kid Friendly Finance" - This book is not primarily about investing for beginners but can help parents teach their kids about money and finance. 

However, based on the content you provided, you may be looking for books about investing. You can look up reviews of various investing books on Amazon and choose one that best suits your needs.
